In [1]:
! mv /Users/spangher/Downloads/comment_dedup.py .

In [3]:
import pandas as pd 
from comment_dedup import CommentDeduplicator

In [4]:
df = pd.read_csv('../data/bulk_downloads/aphis/aphis_2017_2018/public_submission_all_text.csv')

/var/folders/xh/qnyq7yzj0r328_7hnb7pgxth0000gp/T/ipykernel_75803/2820915934.py:1: DtypeWarning: Columns (50,51,56) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/bulk_downloads/aphis/aphis_2017_2018/public_submission_all_text.csv')


In [ ]:
docket = df.loc[lambda df: df['Docket ID'] == 'APHIS-2008-0076'][['Document ID', 'canonical_text']]
input = docket.rename(columns={'Document ID': 'id', 'canonical_text': 'text'}).to_dict(orient='records')
dedup = CommentDeduplicator(threshold=0.8)
dedup.add_comments(input)
dedup.cluster()

In [18]:
dedup.clusters[0].member_ids

['APHIS-2008-0076-0068', 'APHIS-2008-0076-0069']

In [19]:
dedup.clusters[1].member_ids

['APHIS-2008-0076-0076', 'APHIS-2008-0076-0077']

# Identified clusters

In [167]:
import glob
cluster_jsonl_files = glob.glob('../data/bulk_downloads/*/*/public_submission_all_text__dedup_clusters.jsonl')
cluster_mapper_files = glob.glob('../data/bulk_downloads/*/*/public_submission_all_text__dedup_mapper.csv')

In [147]:
cluster_jsonl_files[:5]

['../data/bulk_downloads/msha/msha_2017_2018/public_submission_all_text__dedup_clusters.jsonl',
 '../data/bulk_downloads/msha/msha_2024_2025/public_submission_all_text__dedup_clusters.jsonl',
 '../data/bulk_downloads/msha/msha_2020_2021/public_submission_all_text__dedup_clusters.jsonl',
 '../data/bulk_downloads/msha/msha_2023_2024/public_submission_all_text__dedup_clusters.jsonl',
 '../data/bulk_downloads/msha/msha_2022_2023/public_submission_all_text__dedup_clusters.jsonl']

In [168]:
full_cluster_map = pd.concat(list(map(pd.read_csv, cluster_mapper_files)))

In [171]:
full_cluster_map['cluster_uid'].value_counts()

cluster_uid
FDA-2021-N-1349::24      26964
FDA-2021-N-1349::22      26833
FDA-2015-N-0030::3       23390
FS-2021-0007::231        19867
FDA-2021-N-1349::27      19799
                         ...  
FDA-2018-N-3522::2369        1
FDA-2018-N-3522::2368        1
FDA-2018-N-3522::2367        1
FDA-2018-N-3522::2366        1
FS-2020-0007::755            1
Name: count, Length: 2381490, dtype: int64

In [173]:
full_cluster_map.shape

(4219284, 5)

In [143]:
cluster_mapper_files[11]

'../data/bulk_downloads/cdc/cdc_2018_2019/public_submission_all_text__dedup_mapper.csv'

In [144]:
cluster_mapper = pd.read_csv(cluster_mapper_files[-1])

In [146]:
cluster_uids = cluster_mapper['cluster_uid'].value_counts()
cluster_uids

cluster_uid
EPA-HQ-OA-2018-0259::14      1514
EPA-HQ-OA-2018-0259::12       429
EPA-HQ-OA-2018-0259::44       255
EPA-HQ-OA-2018-0259::46       133
EPA-HQ-OA-2018-0259::49       103
                             ... 
EPA-HQ-OA-2018-0259::7739       1
EPA-HQ-OA-2018-0259::7738       1
EPA-HQ-OA-2018-0259::7737       1
EPA-HQ-OA-2018-0259::7736       1
EPA-R10-UST-2019-0363::2        1
Name: count, Length: 22583, dtype: int64

In [ ]:
all_text_file = cluster_mapper_files[-1].replace('__dedup_mapper.csv', '.csv')
all_text_df = pd.read_csv(all_text_file)

In [155]:
one_cluster = (
    all_text_df
        .loc[lambda df: 
            df['Document ID'].isin(
                cluster_mapper.loc[lambda df: df['cluster_uid'] == cluster_uids.index[0]]['document_id'].tolist())]
)

In [165]:
import pyperclip
idx = 7
pyperclip.copy(one_cluster['canonical_text'].iloc[idx])

# Examine Comment Claims

In [ ]:
import ast
import json

def robust_json(x):
    try:
        return json.loads(x)
    except:
        try:
            return ast.literal_eval(x)
        except:
            return 
    return 

def parse_claim(x):
    output = robust_json(x['claims_parsed_json'])
    if output is None:
        output = robust_json(x['claims_fix_parsed_json'])
    return output
        


In [ ]:
df = df[['agency_id', 'docket_id', 'cluster_uid', 'claims_parsed_json', 'claims_fix_parsed_json']]
df['claims'] = df.apply(parse_claim, axis=1)
df.drop(columns=['claims_parsed_json', 'claims_fix_parsed_json'])